In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install chromadb

In [ ]:
!pip install pandas openai langchain faiss-cpu sentence-transformers

In [ ]:
#!pip uninstall -y langchain langchain-core langchain-openai langchain-community
!pip install langchain langchain-core langchain-openai langchain-community

import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [ ]:
!pip install -U langchain langchain-openai langchain-community langchain-text-splitters chromadb pypdf

In [ ]:
loader = PyPDFLoader("/content/drive/MyDrive/Colab_Notebooks/GENAI/Module9_RAG/Route.pdf")
documents = loader.load()


In [ ]:
# 3. Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
docs = text_splitter.split_documents(documents)

In [ ]:
# 5. Store in Chroma
import os
import shutil # For removing directories

# Uninstall and reinstall chromadb and opentelemetry to resolve persistent dependency conflicts
!pip uninstall -y chromadb opentelemetry-sdk opentelemetry-api opentelemetry-exporter-otlp-proto-grpc opentelemetry-semantic-conventions
!pip install chromadb

# Define key_path directly in this cell to ensure it's available
key_path = '/content/drive/MyDrive/Colab_Notebooks/GENAI/'

chroma_db_path = os.path.join(key_path, "chroma_db")

# Remove existing chroma_db directory if it exists, to ensure a clean start
if os.path.exists(chroma_db_path):
    shutil.rmtree(chroma_db_path)

os.makedirs(chroma_db_path, exist_ok=True) # Ensure directory exists
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding,
    persist_directory=chroma_db_path # Use the explicit path
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

Found existing installation: chromadb 1.5.5
Uninstalling chromadb-1.5.5:
  Successfully uninstalled chromadb-1.5.5
Found existing installation: opentelemetry-sdk 1.40.0
Uninstalling opentelemetry-sdk-1.40.0:
  Successfully uninstalled opentelemetry-sdk-1.40.0
Found existing installation: opentelemetry-api 1.40.0
Uninstalling opentelemetry-api-1.40.0:
  Successfully uninstalled opentelemetry-api-1.40.0
Found existing installation: opentelemetry-exporter-otlp-proto-grpc 1.40.0
Uninstalling opentelemetry-exporter-otlp-proto-grpc-1.40.0:
  Successfully uninstalled opentelemetry-exporter-otlp-proto-grpc-1.40.0
Found existing installation: opentelemetry-semantic-conventions 0.61b0
Uninstalling opentelemetry-semantic-conventions-0.61b0:
  Successfully uninstalled opentelemetry-semantic-conventions-0.61b0
  Using cached chromadb-1.5.5-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.2 kB)
  Using cached opentelemetry_api-1.40.0-py3-none-any.whl.metadata (1.5 kB)
  Using cac

In [ ]:
# 6. LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context below.

Context:
{context}

Question:
{question}
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
)

In [ ]:
response = rag_chain.invoke("What is this document about?")
print(response.content)

This document provides a routelist for a transportation service for the academic year 2025-26, detailing pick-up and estimated drop times for various locations. It includes specific routes and stops, indicating the schedule for students or passengers using the service.


In [ ]:
#User Question
#      ↓
#Vector Search (Top 20)
#      ↓
#Cross-Encoder Reranker (Score pairs)
#      ↓
#Top 5
#      ↓
#LLM Answer

In [ ]:
#pip install -U langchain langchain-openai langchain-community chromadb
#pip install sentence-transformers

In [ ]:
import os
from typing import List

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

from sentence_transformers import CrossEncoder
import numpy as np

In [ ]:
loader = PyPDFLoader("/content/drive/MyDrive/Colab_Notebooks/GENAI/Module9_RAG/Route.pdf")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = splitter.split_documents(docs)

In [ ]:
embedding = OpenAIEmbeddings()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding
)

# Retrieve top 10 initially
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

In [ ]:
#Cross-Encoder Reranker
from sentence_transformers import CrossEncoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [ ]:
###
##But embedding search is approximate.

###
## Takes retrieved docs
## Uses a Cross-Encoder model to rescore them
## Sorts them by actual relevance
## Picks the best 3
## Returns them as final context

###This process is called reranking.
#################

def rerank_with_cross_encoder(input_dict):
    question = input_dict["question"]
    docs: List[Document] = input_dict["docs"]
    #Extract Queation and docs(list of retrieved documents)
    #
    # Create (query, doc) pairs
    pairs = [(question, doc.page_content) for doc in docs]

    # Get relevance scores
    scores = cross_encoder.predict(pairs)
    #This model:

     #Takes full query + full document together

     #Produces a relevance score
    #
    # Sort docs by score descending
    ranked_docs = [
        doc for _, doc in sorted(zip(scores, docs), reverse=True)
    ]

    # Select top 3
    top_docs = ranked_docs[:3]

    return {
        "question": question,
        "context": "\n\n".join(doc.page_content for doc in top_docs)
    }

In [ ]:
##Build Final RAG Chain
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context.

Context:
{context}

Question:
{question}
""")

rag_chain = (
    {
        "docs": retriever,
        "question": RunnablePassthrough()
    }
    | RunnableLambda(rerank_with_cross_encoder)
    | prompt
    | llm
)

In [ ]:
 ##LLM-BASED ReReanker
 ##LLM-Based Reranker
 llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def llm_rerank(input_dict):
    question = input_dict["question"]
    docs: List[Document] = input_dict["docs"]

    scored_docs = []

    for doc in docs:
        scoring_prompt = f"""
        You are a relevance scoring system.

        Question:
        {question}

        Document:
        {doc.page_content}

        Score the document relevance from 1 (irrelevant) to 10 (highly relevant).
        Only return the number.
        """

        response = llm.invoke(scoring_prompt)
        score_text = response.content.strip()

        try:
            score = int(score_text)
        except:
            score = 0

        scored_docs.append((score, doc))

    # Sort highest score first
    scored_docs.sort(key=lambda x: x[0], reverse=True)

    # Select top 3
    top_docs = [doc for _, doc in scored_docs[:3]]

    return {
        "question": question,
        "context": "\n\n".join(d.page_content for d in top_docs)
    }

In [ ]:
response = rag_chain.invoke("what are picks and drops in yalanka")
print(response.content)

In Yelahanka, the pick-up and drop locations are as follows:

- Pick-up: Yelahanka Coffee Day at 07:10
- Estimated Drop Time: 15:35


### RAG with Sentence-based Chunking

Now, let's explore a different chunking strategy: sentence-based chunking. This method splits documents into individual sentences, which can be beneficial for tasks requiring fine-grained context.

In [ ]:
#Example 2

### Create a new ChromaDB with sentence chunks

In [ ]:
# 1. Define a sentence-based text splitter
# We'll use RecursiveCharacterTextSplitter with specific separators for sentences.
text_splitter_sentence = RecursiveCharacterTextSplitter(
    chunk_size=200,  # Smaller chunk size for sentences
    chunk_overlap=50, # Small overlap to maintain some context
    separators=[". ", "! ", "? ", "\n\n", "\n", " ", ""]
)

sentence_docs = text_splitter_sentence.split_documents(documents)

print(f"Number of sentence-based chunks: {len(sentence_docs)}")
print("First few sentence chunks:")
for i, doc in enumerate(sentence_docs[:3]):
    print(f"Chunk {i+1}: {doc.page_content[:150]}...")

# 2. Create a new ChromaDB instance for sentence chunks
sentence_chroma_db_path = os.path.join(key_path, "chroma_db_sentence")

# Remove existing directory if it exists
if os.path.exists(sentence_chroma_db_path):
    shutil.rmtree(sentence_chroma_db_path)

os.makedirs(sentence_chroma_db_path, exist_ok=True)

vectorstore_sentence = Chroma.from_documents(
    documents=sentence_docs,
    embedding=embedding,
    persist_directory=sentence_chroma_db_path
)

retriever_sentence = vectorstore_sentence.as_retriever(search_kwargs={"k": 3})
print("New ChromaDB created with sentence chunks.")

Number of sentence-based chunks: 38
First few sentence chunks:
Chunk 1: Routelist for AY2025-26
Version 1
Pick-up 
Time
Estimated 
Drop Time
1 Adarsh Palm Retreat, Bellandur 06:00 17:00
2 Mahesh Travels, Kadubeesanahalli 0...
Chunk 2: 3 Zonasha Paradiso 06:15 16:20
4 Windmills of the Mind, Whitefield 06:35 16:30
Pick-up 
Time
Estimated 
Drop Time
1 Prestige White Meadows 06:10 16:30...
Chunk 3: 2 Prestige Ozone 06:20 16:45
3 Adarsha Palm meadows (Club house) 06:30 16:50
4 Bright Academy Marathahalli - Cyber Security 06:40 17:00
Pick-up 
Time
...
New ChromaDB created with sentence chunks.


### Setup and run RAG chain with sentence chunks

In [ ]:
# 3. Setup RAG chain with the new sentence-based retriever
rag_chain_sentence = (
    {
        "context": retriever_sentence | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
)

# 4. Invoke a query
print("\nQuerying the RAG chain with sentence chunks...")
response_sentence = rag_chain_sentence.invoke("What is this document about?")
print(response_sentence.content)

print("\nQuerying with a specific question...")
response_sentence_specific = rag_chain_sentence.invoke("What is the first route listed?")
print(response_sentence_specific.content)


Querying the RAG chain with sentence chunks...
The document is a routelist for transportation services for the academic year 2025-26, detailing pick-up and estimated drop-off times for various locations.

Querying with a specific question...
The first route listed is Route 18.


# Task
Add a markdown cell for 'Ex3: RAG with Semantic Chunking', define a `RecursiveCharacterTextSplitter` for semantic chunking, split the `documents` into semantic chunks, and print their count and examples. Then, create a new ChromaDB at `os.path.join(key_path, "chroma_db_semantic")` using these semantic chunks and the `embedding` model, and set up a retriever. Finally, set up and invoke a RAG chain with the semantic chunk retriever, the existing `prompt`, and `llm` using example questions like "What is this document about?" and "What is the first route listed?", and summarize the findings.

## Introduce Semantic Chunking

### Subtask:
Add a markdown cell to introduce 'Ex3: RAG with Semantic Chunking' as a new section.


### Ex3: RAG with Semantic Chunking

## Define Semantic Text Splitter

### Subtask:
Create a new `RecursiveCharacterTextSplitter` instance designed for semantic chunking, prioritizing paragraph and line breaks. Then, split the loaded documents into semantic chunks and display their count and a few examples.


**Reasoning**:
I need to instantiate the `RecursiveCharacterTextSplitter` with the specified parameters for semantic chunking, then split the `documents` and print the count and the first three chunks as instructed.



In [ ]:
text_splitter_semantic = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

semantic_docs = text_splitter_semantic.split_documents(documents)

print(f"Number of semantic chunks: {len(semantic_docs)}")
print("First few semantic chunks:")
for i, doc in enumerate(semantic_docs[:3]):
    print(f"Chunk {i+1}: {doc.page_content[:150]}...")

Number of semantic chunks: 19
First few semantic chunks:
Chunk 1: Routelist for AY2025-26
Version 1
Pick-up 
Time
Estimated 
Drop Time
1 Adarsh Palm Retreat, Bellandur 06:00 17:00
2 Mahesh Travels, Kadubeesanahalli 0...
Chunk 2: 2 Prestige Ozone 06:20 16:45
3 Adarsha Palm meadows (Club house) 06:30 16:50
4 Bright Academy Marathahalli - Cyber Security 06:40 17:00
Pick-up 
Time
...
Chunk 3: 4 Gilma Showroom, Banaswadi 06:40 15:50
5 Horamavu (Ring Road Bus Stop) 06:45 15:45
6 Hosmat Hospital 06:48 16:00
7 Regency Magnum 06:55 16:20
8 Dream...


**Reasoning**:
The previous step failed with an `InternalServerError` during the embedding process, likely due to an intermittent issue with the OpenAI API. To address this, I will re-initialize the `embedding` object to ensure a fresh connection and then retry creating the ChromaDB with the semantic chunks.



In [ ]:
embedding = OpenAIEmbeddings(model="text-embedding-3-small")

# Task
### Ex4: RAG with Reranking

Reranking is a technique used to improve the quality of retrieved documents in a Retrieval Augmented Generation (RAG) system. After an initial retrieval step (e.g., using vector similarity search), a reranker model re-scores the retrieved documents based on their relevance to the query. This helps to promote more relevant documents to the top of the list, even if their initial similarity score was not the highest. This often leads to more accurate and contextually appropriate answers from the Language Model (LLM).

## Introduce RAG with Reranking

### Subtask:
Add a markdown cell to introduce 'Ex4: RAG with Reranking' as a new section, explaining the concept of reranking and its benefits in improving retrieval quality.


### Ex4: RAG with Reranking

Reranking is a technique used to improve the quality of retrieved documents in a Retrieval Augmented Generation (RAG) system. After an initial retrieval step (e.g., using vector similarity search), a reranker model re-scores the retrieved documents based on their relevance to the query. This helps to promote more relevant documents to the top of the list, even if their initial similarity score was not the highest. This often leads to more accurate and contextually appropriate answers from the Language Model (LLM).

## Install Reranking Library

### Subtask:
Install the `sentence-transformers` library, which provides `CrossEncoder` models commonly used for reranking.


**Reasoning**:
The subtask requires installing the `sentence-transformers` library, which is accomplished using a pip install command.



In [ ]:
pip install sentence-transformers

In [ ]:
import os
from typing import List

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

## Setup Reranking Model and Retriever

### Subtask:
Initialize a `CrossEncoder` model for reranking. Then, create a `ContextualCompressionRetriever` using the existing vector store's retriever and the reranking model to implement the reranking step.


**Reasoning**:
I need to import the necessary classes, instantiate the reranking model, create a document compressor pipeline, and then create a contextual compression retriever to implement the reranking step as per the instructions.



# Task
The code in cell `gDJzc7YC8rGo` will be updated to include the definition of `cross_encoder`, the `rerank_with_cross_encoder` function, and the `rag_chain` along with necessary imports and dependencies defined in previous cells (`llm`, `prompt`, `retriever`, `format_docs`). The cell will then be executed to demonstrate the reranking functionality.

## Consolidate Reranking Dependencies

### Subtask:
Move the definition of `cross_encoder`, the `rerank_with_cross_encoder` function, and the `rag_chain` (including necessary imports and `llm`, `prompt`, `retriever`, `format_docs` which are already defined in previous cells) into the selected cell (`gDJzc7YC8rGo`) to make it self-contained and runnable.


**Reasoning**:
To consolidate the reranking dependencies, I will move the specified imports, the `cross_encoder` initialization, the `rerank_with_cross_encoder` function, and the `rag_chain` definition into cell `gDJzc7YC8rGo` to make it self-contained and runnable, as requested.



In [ ]:
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from sentence_transformers import CrossEncoder

# Cross-Encoder Reranker
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank_with_cross_encoder(input_dict):
    question = input_dict["question"]
    docs: List[Document] = input_dict["docs"]

    # Create (query, doc) pairs
    pairs = [(question, doc.page_content) for doc in docs]

    # Get relevance scores
    scores = cross_encoder.predict(pairs)

    # Sort docs by score descending
    ranked_docs = [
        doc for _, doc in sorted(zip(scores, docs), reverse=True)
    ]

    # Select top 3
    top_docs = ranked_docs[:3]

    return {
        "question": question,
        "context": "\n\n".join(doc.page_content for doc in top_docs)
    }

##Build Final RAG Chain
# llm, prompt, retriever, format_docs are assumed to be defined in previous cells

rag_chain = (
    {
        "docs": retriever,
        "question": RunnablePassthrough()
    }
    | RunnableLambda(rerank_with_cross_encoder)
    | prompt
    | llm
)

response = rag_chain.invoke("What is transformer architecture?")
print(response.content)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The context provided does not contain any information about transformer architecture. Therefore, I cannot answer the question based on the given context.


**Reasoning**:
The previous execution successfully consolidated the reranking components into cell `gDJzc7YC8rGo` and ran the RAG chain. The output received is an expected response from the LLM, indicating that the provided document does not contain information about 'transformer architecture', which is not an error in the RAG pipeline itself. Therefore, the task of setting up and running the reranking RAG chain is complete.



In [ ]:
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from sentence_transformers import CrossEncoder

# Cross-Encoder Reranker
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank_with_cross_encoder(input_dict):
    question = input_dict["question"]
    docs: List[Document] = input_dict["docs"]

    # Create (query, doc) pairs
    pairs = [(question, doc.page_content) for doc in docs]

    # Get relevance scores
    scores = cross_encoder.predict(pairs)

    # Sort docs by score descending
    ranked_docs = [
        doc for _, doc in sorted(zip(scores, docs), reverse=True)
    ]

    # Select top 3
    top_docs = ranked_docs[:3]

    return {
        "question": question,
        "context": "\n\n".join(doc.page_content for doc in top_docs)
    }

##Build Final RAG Chain
# llm, prompt, retriever, format_docs are assumed to be defined in previous cells

rag_chain = (
    {
        "docs": retriever,
        "question": RunnablePassthrough()
    }
    | RunnableLambda(rerank_with_cross_encoder)
    | prompt
    | llm
)

response = rag_chain.invoke("What are diff routes availble from Koramangala?")
print(response.content)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The available routes from Koramangala are Route 1, Route 2, Route 3, Route 4, Route 5, Route 6, Route 7, and Route 8.


## Add Markdown Summary

### Subtask:
Add a markdown cell containing the summary of RAG with semantic chunking, including the challenges encountered during ChromaDB creation and the RAG chain invocation.


### Summary of RAG with Semantic Chunking

We attempted to implement RAG with semantic chunking. We defined a `RecursiveCharacterTextSplitter` with specific separators to create semantic chunks from the loaded PDF document. The process involved:

1.  **Creating Semantic Chunks**: The `documents` were successfully split into 19 semantic chunks.
2.  **Creating ChromaDB with Semantic Embeddings**: Attempts to create a new ChromaDB instance (`chroma_db_semantic`) and populate it with embeddings from the `semantic_docs` using `OpenAIEmbeddings` encountered persistent `InternalServerError` from the OpenAI API. This error prevented the successful creation of the vector store and its associated retriever (`retriever_semantic`).
3.  **Setting up and Invoking the RAG Chain**: Due to the failure in creating `retriever_semantic`, the RAG chain could not be successfully set up and invoked. The `NameError` indicated that `retriever_semantic` was not defined.

**Conclusion**: While the document splitting for semantic chunks was successful, the subsequent steps of generating embeddings and storing them in ChromaDB were hindered by intermittent API errors from OpenAI. Therefore, the performance and characteristics of the RAG chain with semantic chunking could not be evaluated at this time. Should the API issues be resolved, the intended outcome was to compare the responses from semantic chunking against previous chunking strategies for relevance and accuracy.


## Summary:

### Data Analysis Key Findings

*   The document was successfully split into 19 semantic chunks using a `RecursiveCharacterTextSplitter`.
*   Attempts to create a ChromaDB instance (`chroma_db_semantic`) and populate it with embeddings using `OpenAIEmbeddings` consistently failed due to `InternalServerError` from the OpenAI API. This prevented the successful creation of the vector store and its associated retriever (`retriever_semantic`).
*   Consequently, the RAG chain could not be successfully set up or invoked, resulting in a `NameError` because `retriever_semantic` was not defined.
*   Despite these technical challenges in setting up the semantic chunking RAG chain, the task of generating and adding a markdown summary detailing these findings to the notebook was successfully completed.

### Insights or Next Steps

*   Investigate and resolve the `InternalServerError` originating from the OpenAI API to enable the successful creation of ChromaDB with semantic embeddings.
*   Once the API issues are addressed and the RAG chain with semantic chunking is functional, evaluate its performance and compare the responses against previous chunking strategies for relevance and accuracy.
